In [48]:
from datasets import load_from_disk
import pandas as pd

In [49]:
ds = load_from_disk("yambda")
pa_table = ds["train"].data.table

df: pd.DataFrame = pd.DataFrame.from_arrow(pa_table)
df

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,100,39420,8326270,0,100.0,170.0,listen
1,100,39420,1441281,0,100.0,105.0,listen
2,100,39625,286361,0,100.0,185.0,listen
3,100,40110,732449,0,100.0,240.0,listen
4,100,40360,3397170,0,46.0,130.0,listen
...,...,...,...,...,...,...,...
47790444,1000000,25961415,3369589,0,99.0,185.0,listen
47790445,1000000,25961615,8120372,0,99.0,200.0,listen
47790446,1000000,25961805,1578810,0,99.0,190.0,listen
47790447,1000000,25962060,3732104,0,100.0,255.0,listen


Мы решили оставить пока только органик данные. Это чисто прослушивания самого пользователя. Когда is_organic == 0, то это трек, который порекоммендовала система. 

In [50]:
df.columns.to_list()

['uid',
 'timestamp',
 'item_id',
 'is_organic',
 'played_ratio_pct',
 'track_length_seconds',
 'event_type']

In [51]:
df['uid'].nunique() 

10000

In [52]:
df['item_id'].nunique()

934057

In [53]:
df['event_type'].unique()

<ArrowStringArray>
['listen', 'like', 'unlike', 'dislike', 'undislike']
Length: 5, dtype: str

In [56]:
df_organic = df[df['is_organic'] == 1].drop(['is_organic'], axis=1)

In [57]:
df_organic

,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,event_type
34,100,44755,732449,NaN,NaN,like
57,100,104725,732449,1.0,240.0,listen
58,100,287910,5699545,100.0,120.0,listen
120,100,1259125,5411243,NaN,NaN,like
155,100,1265850,4943655,100.0,165.0,listen
...,...,...,...,...,...,...
47790432,1000000,25958915,2317630,100.0,270.0,listen
47790433,1000000,25959095,7166646,99.0,180.0,listen
47790434,1000000,25959395,8405281,100.0,300.0,listen
47790435,1000000,25959625,3334920,100.0,230.0,listen


Я решил попробовать в начале методы, которые хорошо работают только с бинарными матрицами интеракций. После этого я хочу построить взвешенную матрицу взаимодействий и уже на них эксперементировать.

Идея для взвешивания такова: у нас есть played ratio у трека и тип взаимодействия. Относительно этого можно попробовать смоделировать более точное отношение пользователя к треку

Как я буду конструировать бинарные признаки взаимодействия для матрицы интеракций:

* Если у нас есть explicit фидбек, по типу like/dislike, то это 0 или 1 соответственно;
* Если у нас implicit фидбек, а именно ивент = listen, то тут я смотрю на `played_ratio_pct`. Если ratio оказалось больше 50, то есть трек на половину послушали, то это 1, иначе 0;
* Я буду учитывать время последнего взаимодействия с треком. То есть самый последний ивент в итоге и будет учтен. Если, например, у нас было undislike, то трек становится нейтральным - то есть listen, то тут подключается правильно с `played_ratio_pct`. Если после этого его еще раз дизлайкнут, то в итоге в матрице останется 0.

Данные буду делить при помощи стратегии Global Time Split

In [59]:
df_organic['timestamp'].min(), df_organic['timestamp'].max()

(np.uint32(15), np.uint32(26000000))

In [60]:
df_organic['uid'].nunique(), df_organic['item_id'].nunique()

(9677, 771519)

Давайте рассмотрим, для примера, самого активного юзера и историю его взаимодействия с айтемами

In [78]:
df_organic['uid'].mode()

0    702000
Name: uid, dtype: uint32

In [81]:
uid_702000 = df_organic[(df_organic['uid'] == 702000)]

In [82]:
uid_702000

,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,event_type
33643323,702000,63170,3982464,20.0,215.0,listen
33643324,702000,63400,6726825,50.0,450.0,listen
33643325,702000,251055,5318084,58.0,160.0,listen
33643326,702000,251060,408251,2.0,210.0,listen
33643327,702000,251075,3160438,7.0,175.0,listen
...,...,...,...,...,...,...
33670358,702000,25999565,5456928,100.0,175.0,listen
33670359,702000,25999680,1684982,100.0,115.0,listen
33670360,702000,25999795,646738,36.0,315.0,listen
33670361,702000,25999815,6572094,10.0,180.0,listen


In [85]:
uid_702000[uid_702000['item_id'] == 7271859].sort_values(by='timestamp')

,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,event_type
33651371,702000,9745695,7271859,NaN,NaN,like
33651372,702000,9745695,7271859,NaN,NaN,undislike
33651377,702000,9749845,7271859,1.0,210.0,listen
33651538,702000,9804840,7271859,11.0,210.0,listen
33651570,702000,9832720,7271859,0.0,210.0,listen
33652876,702000,11494680,7271859,0.0,210.0,listen
33653774,702000,12255180,7271859,0.0,210.0,listen
33653817,702000,12442430,7271859,100.0,210.0,listen
33654549,702000,14001835,7271859,0.0,210.0,listen
33654928,702000,14524505,7271859,100.0,210.0,listen


In [86]:
uid_702000[uid_702000['item_id'] == 3160438].sort_values(by='timestamp')

,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,event_type
33643327,702000,251075,3160438,7.0,175.0,listen
33643468,702000,867120,3160438,2.0,175.0,listen
33643500,702000,972865,3160438,84.0,175.0,listen
33643519,702000,1026715,3160438,100.0,175.0,listen
33643554,702000,1102335,3160438,64.0,175.0,listen
33643555,702000,1102400,3160438,35.0,175.0,listen
33643906,702000,1718325,3160438,100.0,175.0,listen
33644238,702000,1980460,3160438,42.0,175.0,listen
33644428,702000,2177300,3160438,49.0,175.0,listen
33645024,702000,2768860,3160438,100.0,175.0,listen


In [88]:
uid_702000[uid_702000['item_id'] == 646738].sort_values(by='timestamp')

,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,event_type
33670360,702000,25999795,646738,36.0,315.0,listen


In [ ]:
uid_702000[uid_702000['item_id'] == 6572094].sort_values(by='timestamp')

,uid,timestamp,item_id,played_ratio_pct,track_length_seconds,event_type
33669804,702000,25383350,6572094,100.0,180.0,listen
33669814,702000,25384785,6572094,27.0,180.0,listen
33669816,702000,25384865,6572094,NaN,NaN,like
33669817,702000,25385025,6572094,100.0,180.0,listen
33669820,702000,25385100,6572094,19.0,180.0,listen
33669825,702000,25385205,6572094,13.0,180.0,listen
33669829,702000,25385690,6572094,5.0,180.0,listen
33669844,702000,25389755,6572094,100.0,180.0,listen
33669874,702000,25395235,6572094,52.0,180.0,listen
33669875,702000,25395420,6572094,100.0,180.0,listen


Если в истории взаимодействия есть лайк, то давайте считать это 1.

Если пользователь только слушал трек, то мы смотрим на среднее по `played_ratio_pct`

Если у нас появляется что-то непонятное, по типу: like -> undislike, то мы считаем, что в итоге пользователь имеет лайк. Попробуем смоделировать адекватное взаимодействие:

* like -> unlike
* like -> dislike
* dislike -> undislike
* dislike -> like
* undislike -> dislike
* undislike -> like

Иная последовательность, по типу: like -> undislike считается невалидной и мы оставляем только like.

Теперь давайте поделим данные, а потом сделаем агрегацию по времени

In [92]:
df_organic_sorted = df_organic.sort_values(by=['uid', 'item_id', 'timestamp'])

explicit_events = ['like', 'unlike', 'dislike', 'undislike']
df_explicit = df_organic_sorted[df_organic_sorted['event_type'].isin(explicit_events)]

final_states = df_explicit.drop_duplicates(subset=['uid', 'item_id'], keep='last').copy()

weight_mapping = {
    'like': 1,
    'unlike': 0,
    'dislike': 0,
    'undislike': 0
}
final_states['interaction_weight'] = final_states['event_type'].map(weight_mapping)

user_item_weights = final_states[['uid', 'item_id', 'interaction_weight']]

In [93]:
user_item_weights

,uid,item_id,interaction_weight
264,100,410868,1
34,100,732449,1
287,100,1730273,1
3162,100,2263048,1
3303,100,3526521,1
...,...,...,...
47789147,999900,8974272,0
47788285,999900,9125643,1
47789979,1000000,3694666,1
47789980,1000000,5064764,0
